In [3]:
import pandas as pd

df = pd.read_csv('../data/processed/games_clean.csv')

df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])

print(df.dtypes)

SEASON_ID                     int64
TEAM_ID                       int64
TEAM_ABBREVIATION               str
TEAM_NAME                       str
GAME_ID                       int64
GAME_DATE            datetime64[us]
MATCHUP                         str
WL                              str
MIN                           int64
PTS                           int64
FGM                           int64
FGA                           int64
FG_PCT                      float64
FG3M                          int64
FG3A                          int64
FG3_PCT                     float64
FTM                           int64
FTA                           int64
FT_PCT                      float64
OREB                          int64
DREB                          int64
REB                           int64
AST                           int64
STL                           int64
BLK                           int64
TOV                           int64
PF                            int64
PLUS_MINUS                  

Rest days and back to back

In [19]:
df_sorted = df.sort_values(['TEAM_ABBREVIATION', 'GAME_DATE'])

df_sorted['rest_days'] = df_sorted.groupby(['TEAM_ABBREVIATION', 'SEASON_ID'])['GAME_DATE'].diff().dt.days

median_rest = df_sorted['rest_days'].median()
df_sorted['rest_days'] = df_sorted['rest_days'].fillna(median_rest)

df_sorted['is_back_to_back'] = (df_sorted['rest_days'] == 1).astype(int)

print(df_sorted[['TEAM_ABBREVIATION', 'SEASON_ID', 'GAME_DATE', 'rest_days', 'is_back_to_back']].head(15))

     TEAM_ABBREVIATION  SEASON_ID  GAME_DATE  rest_days  is_back_to_back
2138               ATL      22020 2020-12-23        2.0                0
2110               ATL      22020 2020-12-26        3.0                0
2074               ATL      22020 2020-12-28        2.0                0
2047               ATL      22020 2020-12-30        2.0                0
2007               ATL      22020 2021-01-01        2.0                0
2004               ATL      22020 2021-01-02        1.0                1
1968               ATL      22020 2021-01-04        2.0                0
1947               ATL      22020 2021-01-06        2.0                0
1882               ATL      22020 2021-01-09        3.0                0
1861               ATL      22020 2021-01-11        2.0                0
1809               ATL      22020 2021-01-15        4.0                0
1798               ATL      22020 2021-01-16        1.0                1
1779               ATL      22020 2021-01-18       

In [20]:
print(df_sorted.value_counts('is_back_to_back'))

is_back_to_back
0    11866
1     2594
Name: count, dtype: int64


In [16]:
print(df_sorted[(df_sorted['TEAM_ABBREVIATION'] == 'ATL')].groupby('SEASON_ID').head(1)[
    ['SEASON_ID', 'GAME_DATE', 'rest_days']])

       SEASON_ID  GAME_DATE  rest_days
2138       22020 2020-12-23        2.0
4593       22021 2021-10-21        2.0
7073       22022 2022-10-19        2.0
9533       22023 2023-10-25        2.0
11986      22024 2024-10-23        2.0
14437      22025 2025-10-22        2.0


Rolling averages

In [22]:
stat_cols = ['PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'FG_PCT', 'FG3_PCT']

window=5

for col in stat_cols:
    df_sorted[f'rolling_{col.lower()}_{window}'] = df_sorted.groupby(['TEAM_ABBREVIATION', 'SEASON_ID'])[col].shift(1).rolling(window=window, min_periods=1).mean()



In [24]:
print(df_sorted[['TEAM_ABBREVIATION', 'PTS', 'rolling_pts_5']].head(15))

     TEAM_ABBREVIATION  PTS  rolling_pts_5
2138               ATL  124            NaN
2110               ATL  122     124.000000
2074               ATL  128     123.000000
2047               ATL  141     124.666667
2007               ATL  114     128.750000
2004               ATL   91     125.800000
1968               ATL  108     119.200000
1947               ATL   94     116.400000
1882               ATL  105     109.600000
1861               ATL  112     102.400000
1809               ATL   92     102.000000
1798               ATL  106     102.200000
1779               ATL  108     101.800000
1747               ATL  123     104.600000
1731               ATL  116     108.200000
